# Fairness

**Goal:**
In this section, we assess the fairness of our classifier regarding two protected attributes: **Sex** and **Age**.
Machine Learning models can unintentionally learn historical biases present in the data (e.g., if women were historically paid less, the model might learn to predict lower income for women).

**Methodology:**
1.  **Metric:** We will use the **Statistical Parity Difference (SPD)**. This metric measures the difference in the probability of receiving a positive outcome (High Income) between the unprivileged group and the privileged group.
    * $SPD = P(Y=1 | Unprivileged) - P(Y=1 | Privileged)$
    * A value of 0 implies perfect fairness.
    * A negative value implies bias against the unprivileged group.
2.  **Mitigation:** We will apply a pre-processing technique called **Reweighing**. Instead of modifying the feature values, this algorithm assigns different **weights** to examples in the training data to ensure that protected groups are represented equally in the positive outcome class.

## Installation and Imports

In [2]:
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing

# We need to reconstruct the training data format for AIF360
# We combine X_train and y_train back into a dataframe temporarily
train_df = X_train.copy()
train_df['Income'] = y_train.copy()

# Define which values are privileged
# Sex: 1 (Male) is privileged, 0 (Female) is unprivileged
# Age: 1 (Senior > Median) is privileged (usually earn more), 0 (Young) is unprivileged
privileged_groups = [{'Sex': 1}]
unprivileged_groups = [{'Sex': 0}]

print("Fairness libraries imported and data prepared.")

NameError: name 'X_train' is not defined

## Assess Fairness (Before Mitigation)

First, we transform our Pandas DataFrames into `BinaryLabelDataset` objects, which are required by the `aif360` library.
Then, we calculate the **Statistical Parity Difference** on the original training data to see if there is bias regarding Sex.

In [ ]:
# Convert the Pandas DataFrame to an AIF360 BinaryLabelDataset
dataset_orig_train = BinaryLabelDataset(
    favorable_label=1,
    unfavorable_label=0,
    df=train_df,
    label_names=['Income'],
    protected_attribute_names=['Sex', 'Age']
)

# Compute the metric for the original dataset
metric_orig_train = BinaryLabelDatasetMetric(
    dataset_orig_train, 
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

spd_orig = metric_orig_train.statistical_parity_difference()

print(f"Original Training Data - Statistical Parity Difference (Sex): {spd_orig:.4f}")

if spd_orig < -0.1:
    print(">> Bias detected: The unprivileged group (Female) is significantly less likely to have high income.")
else:
    print(">> The dataset seems relatively fair.")